# Cease & Desist Multi-Agent System (LangGraph + Groq)
### With PDF + OCR + Batch Processing
Processes all PDFs in a folder and classifies them.

In [1]:
!pip install langchain langgraph groq pydantic python-dotenv pymupdf pytesseract pillow

In [2]:
import os
from typing import TypedDict, List
from datetime import datetime
from langgraph.graph import StateGraph
import sqlite3
from groq import Groq
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
from IPython.display import display, clear_output


/Users/sarchana/Documents/CapstoneProject/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
os.environ['GROQ_API_KEY'] = "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"
client = Groq(api_key=os.environ['GROQ_API_KEY'])
PDF_DIR = "/Users/sarchana/Documents/CapstoneProject/documents"
ARCHIVE_DIR = os.path.join(PDF_DIR, "archive")
os.makedirs(ARCHIVE_DIR, exist_ok=True)
SUPPORTED_EXTENSIONS = ('.pdf', '.jpg', '.jpeg', '.png')

In [4]:
class State(TypedDict):
    document_name: str
    document_text: str
    received_date: str
    classification: str
    confidence: float
    final_action: str
    agent_path: List[str]
    human_decision: str
    audit_logged: bool
    archieve_path: str


In [5]:
def extract_text_from_pdf(file_path):
    try:
        doc = fitz.open(file_path)
        full_text = ''
        for page in doc:
            text = page.get_text()
            if text.strip():
                full_text += text
            else:
                pix = page.get_pixmap()
                img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
                full_text += pytesseract.image_to_string(img)
        return full_text
    
    except Exception as e:
        print("Error while extracting text for - {file_path}")
        return ""


In [6]:
db = []

def classification_agent(state):
    prompt = f"Classify as Cease, Irrelevant or Uncertain. If text provided is blank classify that file in to Uncertain: {state['document_text']}"
    resp = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        #model='llama-3.3-70b-versatile',
        messages=[{'role':'user','content':prompt}],
        temperature=0.2
    )
    text = resp.choices[0].message.content.lower()
    if 'cease' in text:
        state['classification']='Cease'
    elif 'irrelevant' in text:
        state['classification']='Irrelevant'
    else:
        state['classification']='Uncertain'
    state['confidence']=0.8
    state['agent_path'].append('classification')
    return state



def database_agent(state):
    conn = sqlite3.connect("cease_desist.db")

    conn.execute("""
     CREATE TABLE IF NOT EXISTS cease_requests (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            doc_name TEXT,
            date TEXT,
            details TEXT
        )
    """)

    conn.execute(
        "INSERT INTO cease_requests (doc_name, date, details) VALUES (?, ?, ?)",
        (state['document_name'], state['received_date'], state['document_text'])
    )

    conn.commit()
    conn.close()
    db.append(state)
    state['final_action'] = 'stored'
    state['agent_path'].append('database')
    return state


def archiving_agent(state):
    #with open('archive_log.txt','a') as f:
     #   f.write(state['document_name']+'\n')
    #state['final_action']='archived'
    #state['agent_path'].append('archive')
    #return state

    try:

            # ✅ Unique file name
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            archive_name = f"{timestamp}_{state['document_name']}.txt"

            archive_path = os.path.join(ARCHIVE_DIR, archive_name)

            # ✅ Write structured text file
            with open(archive_path, "w", encoding="utf-8") as f:
                f.write("========== DOCUMENT ARCHIVE ==========\n\n")
                f.write(f"Original File : {state['document_name']}\n")
                f.write(f"Archived At   : {datetime.now()}\n")
                f.write("\n========== EXTRACTED TEXT ==========\n\n")
                f.write(state['document_text'].strip())

            print(f"📄 Archived as TXT → {archive_path}")
            state['archieve_path']=archive_path

            return state

    except Exception as e:
            print(f"❌ Archive Error: {e}")
            return None



def hitl_agent(state):
    # Display info
        print("\n⚠️ HUMAN REVIEW REQUIRED")
        print(f"📄 Document: {state['document_name']}")
        print("-" * 60)

        preview = state['document_text'][:300].replace("\n", " ")
        print(f"📝 Preview: {preview}...")
        print("-" * 60)

        if state['classification']:
            print(f"🤖 Model Suggestion: {state['classification']}")

        decision = input(f"Manual review needed for {state['document_name']} (Cease/Irrelevant): ")
        state['human_decision']=decision
        state['agent_path'].append('hitl')
        return state

def audit_agent(state):
    print('Audit:', state['document_name'], state['classification'], state['final_action'])
    with open("audit_log.txt","a",encoding="utf-8") as f : f.write(f"Audit: {state['document_name']} ---{state['classification']}----{state['final_action']}-----{state['human_decision']}\n")
    state['audit_logged']=True
    state['agent_path'].append('audit')
    return state


In [7]:
builder = StateGraph(State)

builder.add_node('classify', classification_agent)
builder.add_node('database', database_agent)
builder.add_node('archive', archiving_agent)
builder.add_node('hitl', hitl_agent)
builder.add_node('audit', audit_agent)

def route(state):
    if state['classification']=='Cease': return 'database'
    elif state['classification']=='Irrelevant': return 'archive'
    else: return 'hitl'

def route_hitl(state):
    if state['human_decision']=='Cease': return 'database'
    else: return 'archive'

builder.add_conditional_edges('classify', route)
builder.add_conditional_edges('hitl', route_hitl)

builder.add_edge('database','audit')
builder.add_edge('archive','audit')

builder.set_entry_point('classify')
graph = builder.compile()

In [8]:


for file in os.listdir(PDF_DIR):
    if file.endswith(SUPPORTED_EXTENSIONS):
        path = os.path.join(PDF_DIR, file)
        print('File - ',file)
        text = extract_text_from_pdf(path)

        state = {
            'document_name': file,
            'document_text': text,
            'received_date': str(datetime.now()),
            'classification':'',
            'confidence':0.0,
            'final_action':'',
            'agent_path':[],
            'human_decision':'',
            'audit_logged':False,
            'archieve_path':''
        }

        result = graph.invoke(state)
        print('Processed:', file, '->', result['classification'])


File -  20251009_111416.jpg
Audit: 20251009_111416.jpg Cease stored
Processed: 20251009_111416.jpg -> Cease
File -  LOA3.pdf
Audit: LOA3.pdf Cease stored
Processed: LOA3.pdf -> Cease
File -  LOA2.pdf
Audit: LOA2.pdf Cease stored
Processed: LOA2.pdf -> Cease
File -  20251009_094233.jpg
Audit: 20251009_094233.jpg Cease stored
Processed: 20251009_094233.jpg -> Cease
File -  LoA1.pdf
Audit: LoA1.pdf Cease stored
Processed: LoA1.pdf -> Cease
File -  20251009_105307.jpg

⚠️ HUMAN REVIEW REQUIRED
📄 Document: 20251009_105307.jpg
------------------------------------------------------------
📝 Preview: ‘t Technology dq  1726521 2193429  1685384 | SreerajV ~-  es Technology  ...
------------------------------------------------------------
🤖 Model Suggestion: Uncertain


Manual review needed for 20251009_105307.jpg (Cease/Irrelevant):  Cease


Audit: 20251009_105307.jpg Uncertain stored
Processed: 20251009_105307.jpg -> Uncertain
File -  LOA5.pdf
Audit: LOA5.pdf Cease stored
Processed: LOA5.pdf -> Cease
File -  LOA4.pdf
Audit: LOA4.pdf Cease stored
Processed: LOA4.pdf -> Cease
File -  20251009_105300.jpg

⚠️ HUMAN REVIEW REQUIRED
📄 Document: 20251009_105300.jpg
------------------------------------------------------------
📝 Preview: Day-1  s a eae!  =  Rajeev i. )  Puttaswamy,  re 1934712 CF 2065717  | 2081617  Tharun Kumar Reddy Kunduru :  a  1991099 | inaMP oe  | 2019399  Neeladri Dutta  Rupesh Kumar  Anand Bhat .  Rameshwar Sharma ey  ADARSH SOMASEKHARAN NAIR - |. Technology  . ...
------------------------------------------------------------
🤖 Model Suggestion: Uncertain


Manual review needed for 20251009_105300.jpg (Cease/Irrelevant):  Cease


Audit: 20251009_105300.jpg Uncertain stored
Processed: 20251009_105300.jpg -> Uncertain
File -  LOA6.pdf
Audit: LOA6.pdf Cease stored
Processed: LOA6.pdf -> Cease
File -  LOA7.pdf
Audit: LOA7.pdf Cease stored
Processed: LOA7.pdf -> Cease
File -  20251009_111058.jpg

⚠️ HUMAN REVIEW REQUIRED
📄 Document: 20251009_111058.jpg
------------------------------------------------------------
📝 Preview: fa an medical eas this authority is acid I (we) may have given to any health  RAAT 956 odeS9 8 Sege70/DEF =) .47.0-W 1108 O/DRN-  ...
------------------------------------------------------------
🤖 Model Suggestion: Uncertain


...

Audit: bw_doc_5.pdf Uncertain stored
Processed: bw_doc_5.pdf -> Uncertain
File -  bw_doc_4.pdf

⚠️ HUMAN REVIEW REQUIRED
📄 Document: bw_doc_4.pdf
------------------------------------------------------------
📝 Preview: Grafton & Meyers, Family Law Division en  Cary Cmeinon Rpt Bl Minar  acre who mn une eng psn gratia te mas ca aneng oman nny eae Thn cot se mage wt yeti mea other ey scent grr eect  ‘stn ealene or elation lym ‘Sent pt coon hou dag ern pon (Senn he mann  ...
------------------------------------------------------------
🤖 Model Suggestion: Uncertain


Manual review needed for bw_doc_4.pdf (Cease/Irrelevant):  Irrelevant


📄 Archived as TXT → /Users/sarchana/Documents/CapstoneProject/documents/archive/20260323_211649_bw_doc_4.pdf.txt
Audit: bw_doc_4.pdf Uncertain 
Processed: bw_doc_4.pdf -> Uncertain
File -  20251009_110821.jpg
Audit: 20251009_110821.jpg Cease stored
Processed: 20251009_110821.jpg -> Cease
File -  notice_5.pdf

⚠️ HUMAN REVIEW REQUIRED
📄 Document: notice_5.pdf
------------------------------------------------------------
📝 Preview: Merrit & Cole, Attorneys —  ured he  ‘win yn tev youl ion po waren, as pro ‘Stride at pope he Sa a  ...
------------------------------------------------------------
🤖 Model Suggestion: Uncertain


Manual review needed for notice_5.pdf (Cease/Irrelevant):  Irrelevant


📄 Archived as TXT → /Users/sarchana/Documents/CapstoneProject/documents/archive/20260323_211659_notice_5.pdf.txt
Audit: notice_5.pdf Uncertain 
Processed: notice_5.pdf -> Uncertain
File -  20251009_110503.jpg
Audit: 20251009_110503.jpg Cease stored
Processed: 20251009_110503.jpg -> Cease
File -  20251009_111026.jpg
Audit: 20251009_111026.jpg Cease stored
Processed: 20251009_111026.jpg -> Cease
File -  20251009_111232.jpg
Audit: 20251009_111232.jpg Cease stored
Processed: 20251009_111232.jpg -> Cease
File -  LOA9.pdf
Audit: LOA9.pdf Cease stored
Processed: LOA9.pdf -> Cease
File -  LOA8.pdf
Audit: LOA8.pdf Cease stored
Processed: LOA8.pdf -> Cease
File -  20251009_110659.jpg
Audit: 20251009_110659.jpg Cease stored
Processed: 20251009_110659.jpg -> Cease
